# Pipeline Engine Test Suite

This notebook provides an interactive way to:
1. **Set up** test Conda environments
2. **Run** pipeline tests
3. **Clean up** test environments

It's equivalent to running the scripts `setup_environments.py`, `run_tests.py`, and `cleanup_environments.py`, but with detailed explanations and the ability to run tests step-by-step.

---

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Environment Management](#2-environment-management)
   - [Check Conda Status](#21-check-conda-status)
   - [Create Test Environments](#22-create-test-environments)
   - [Verify Environments](#23-verify-environments)
3. [Running Tests](#3-running-tests)
   - [Test 1: Local Execution](#31-test-1-local-execution)
   - [Test 2: Mixed Steps with Data Flow](#32-test-2-mixed-steps-with-data-flow)
   - [Test 3: Step-Level Environment Switching](#33-test-3-step-level-environment-switching)
   - [Test 4: Pipeline-Level Environment Switching](#34-test-4-pipeline-level-environment-switching)
   - [Test 5: Combined Environment Switching](#35-test-5-combined-environment-switching)
   - [Run All Tests](#36-run-all-tests)
4. [Cleanup](#4-cleanup)
5. [Troubleshooting](#5-troubleshooting)

---

## 1. Setup & Imports

First, we need to set up the Python path so we can import the engine, and import the necessary modules.

In [ ]:
# Standard library imports
import os
import sys
import subprocess
import json
from pathlib import Path
from datetime import datetime

# Determine directory paths
# This notebook is in: pipeline_engine/workflows/test1/scripts/
NOTEBOOK_DIR = Path.cwd()
SCRIPTS_DIR = NOTEBOOK_DIR  # Same as notebook location
TEST1_DIR = SCRIPTS_DIR.parent
WORKFLOWS_DIR = TEST1_DIR.parent
ROOT_DIR = WORKFLOWS_DIR.parent  # pipeline_engine/
ENGINE_DIR = ROOT_DIR / "engine"

# Add engine to Python path
if str(ENGINE_DIR) not in sys.path:
    sys.path.insert(0, str(ENGINE_DIR))

print("Directory Structure:")
print(f"  Root:      {ROOT_DIR}")
print(f"  Engine:    {ENGINE_DIR}")
print(f"  Workflows: {WORKFLOWS_DIR}")
print(f"  Test1:     {TEST1_DIR}")
print(f"  Scripts:   {SCRIPTS_DIR}")
print()
print(f"Python: {sys.version}")
print(f"Current Environment: {os.path.basename(sys.prefix)}")

In [ ]:
# Import the pipeline engine
try:
    from engine import run_pipeline
    print("✓ Successfully imported run_pipeline from engine")
except ImportError as e:
    print(f"✗ Failed to import engine: {e}")
    print("  Make sure you're running this notebook from the scripts/ directory")

### Helper Functions

These utility functions are used throughout the notebook for formatted output and running shell commands.

In [ ]:
def print_header(title: str, char: str = "="):
    """Print a formatted header."""
    print(f"\n{char * 60}")
    print(f"  {title}")
    print(f"{char * 60}")

def print_result(name: str, success: bool, details: str = ""):
    """Print a test result."""
    status = "✓ PASS" if success else "✗ FAIL"
    print(f"\n  {status}: {name}")
    if details:
        print(f"          {details}")

def run_command(cmd: list, capture: bool = True) -> subprocess.CompletedProcess:
    """Run a shell command and return the result."""
    print(f"  $ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=capture, text=True)
    return result

print("✓ Helper functions defined")

---

## 2. Environment Management

The pipeline engine can run steps in different Conda environments. For testing, we create three environments:

| Environment | Python | Purpose |
|-------------|--------|--------|
| `env_a` | 3.10 | Primary test environment |
| `env_b` | 3.11 | Secondary environment for switching tests |
| `env_c` | 3.12 | Tertiary environment for nested switching |

### 2.1 Check Conda Status

First, let's verify Conda is available and check its configuration.

In [ ]:
def get_conda_info():
    """Get Conda configuration information."""
    try:
        result = subprocess.run(
            ["conda", "info", "--json"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            return json.loads(result.stdout)
    except FileNotFoundError:
        print("✗ Conda not found in PATH")
        print("  Please install Conda or run from a Conda-enabled terminal")
    except Exception as e:
        print(f"✗ Error getting Conda info: {e}")
    return None

# Check Conda
conda_info = get_conda_info()

if conda_info:
    print("✓ Conda is available")
    print(f"  Version: {conda_info.get('conda_version', 'Unknown')}")
    print(f"  Root prefix: {conda_info.get('root_prefix', 'Unknown')}")
    print(f"  Active environment: {conda_info.get('active_prefix_name', 'None')}")
    print(f"  Platform: {conda_info.get('platform', 'Unknown')}")
else:
    print("\n⚠ Conda is not available.")
    print("  Environment switching tests will not work.")
    print("  Local execution tests will still work.")

### 2.2 Create Test Environments

This creates the three test Conda environments. Each environment includes:
- A specific Python version
- PyYAML (required by the engine)

**Note:** This may take several minutes.

In [ ]:
# Environment definitions
ENVIRONMENTS = {
    "env_a": {"python": "3.10", "packages": ["pyyaml"]},
    "env_b": {"python": "3.11", "packages": ["pyyaml"]},
    "env_c": {"python": "3.12", "packages": ["pyyaml"]},
}

def env_exists(env_name: str) -> bool:
    """Check if a Conda environment exists."""
    conda_info = get_conda_info()
    if not conda_info:
        return False
    
    envs = conda_info.get("envs", [])
    for env_path in envs:
        if Path(env_path).name == env_name:
            return True
    return False

def create_environment(env_name: str, python_version: str, packages: list) -> bool:
    """Create a Conda environment."""
    print(f"\n  Creating {env_name} (Python {python_version})...")
    
    if env_exists(env_name):
        print(f"    → Already exists, skipping")
        return True
    
    # Create environment
    cmd = ["conda", "create", "-n", env_name, f"python={python_version}", "-y", "-q"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"    ✗ Failed to create: {result.stderr[:200]}")
        return False
    
    # Install packages
    if packages:
        print(f"    Installing packages: {', '.join(packages)}")
        cmd = ["conda", "run", "-n", env_name, "pip", "install"] + packages + ["-q"]
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"    ⚠ Warning: Package installation issues")
    
    print(f"    ✓ Created {env_name}")
    return True

print("Environment definitions loaded.")
print("Run the next cell to create the environments.")

In [ ]:
# Create all test environments
# ⚠ This may take several minutes!

print_header("Creating Test Environments")

if not get_conda_info():
    print("\n✗ Conda not available. Cannot create environments.")
else:
    success_count = 0
    
    for env_name, config in ENVIRONMENTS.items():
        if create_environment(env_name, config["python"], config["packages"]):
            success_count += 1
    
    print_header("Setup Complete")
    print(f"\n  Created: {success_count}/{len(ENVIRONMENTS)} environments")
    
    if success_count == len(ENVIRONMENTS):
        print("\n  ✓ All environments ready!")

### 2.3 Verify Environments

Let's check which environments exist and their Python versions.

In [ ]:
def check_environments():
    """Check status of all test environments."""
    print_header("Environment Status")
    
    conda_info = get_conda_info()
    if not conda_info:
        print("\n  ✗ Conda not available")
        return {}
    
    available = {}
    
    for env_name in ENVIRONMENTS:
        exists = env_exists(env_name)
        
        if exists:
            # Get Python version
            result = subprocess.run(
                ["conda", "run", "-n", env_name, "python", "--version"],
                capture_output=True, text=True, timeout=30
            )
            version = result.stdout.strip() or result.stderr.strip()
            print(f"  ✓ {env_name}: {version}")
            available[env_name] = True
        else:
            print(f"  ✗ {env_name}: NOT FOUND")
            available[env_name] = False
    
    return available

env_status = check_environments()

---

## 3. Running Tests

Now let's run the pipeline tests. Each test verifies a different aspect of the engine:

| Test | What it Tests |
|------|---------------|
| **Local Execution** | Steps run in the main process |
| **Mixed Steps** | Data flows between multiple steps |
| **Step-Level Env** | Individual steps can switch environments |
| **Pipeline-Level Env** | Entire pipeline runs in specified environment |
| **Combined** | Nested environment switching |

### Test Configuration

In [ ]:
# Paths to test pipelines
PIPELINES_DIR = TEST1_DIR / "pipelines"

PIPELINE_FILES = {
    "local": PIPELINES_DIR / "test_local_pipeline.yaml",
    "mixed": PIPELINES_DIR / "test_mixed_pipeline.yaml",
    "step_env": PIPELINES_DIR / "test_step_env_pipeline.yaml",
    "pipeline_env": PIPELINES_DIR / "test_pipeline_env_pipeline.yaml",
    "combined": PIPELINES_DIR / "test_combined_env_pipeline.yaml",
}

print("Pipeline files:")
for name, path in PIPELINE_FILES.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {name}: {path.name}")

# Store test results
test_results = {}

### 3.1 Test 1: Local Execution

This test verifies that steps with `environment: "local"` run in the main process (same PID).

**What happens:**
1. Engine loads the YAML file
2. Engine loads and executes `step_local.py`
3. Step runs in the same process as this notebook
4. Step records its process ID in the result

In [ ]:
print_header("Test 1: Local Execution")

main_pid = os.getpid()
main_env = os.path.basename(sys.prefix)

print(f"\n  Main process PID: {main_pid}")
print(f"  Main environment: {main_env}")
print(f"  Pipeline: {PIPELINE_FILES['local'].name}")

try:
    print("\n  Running pipeline...")
    result = run_pipeline(
        yaml_path=str(PIPELINE_FILES['local']),
        label="test_local",
        input_data={"source": "notebook"}
    )
    
    # Check results
    step_data = result.get("step_local", {})
    step_pid = step_data.get("process_id")
    step_env = step_data.get("environment_name")
    
    print(f"\n  Results:")
    print(f"    Step PID: {step_pid}")
    print(f"    Step environment: {step_env}")
    print(f"    Same process: {step_pid == main_pid}")
    print(f"    Workflow name: {result['metadata']['workflow_name']}")
    
    success = step_pid == main_pid
    test_results['local'] = success
    print_result("Local Execution", success, 
                 f"PID match: {main_pid}" if success else f"PIDs differ: {main_pid} vs {step_pid}")
    
except Exception as e:
    test_results['local'] = False
    print_result("Local Execution", False, str(e))

### 3.2 Test 2: Mixed Steps with Data Flow

This test verifies that data flows correctly between multiple steps.

**What happens:**
1. `step_local` runs and adds data to `pipeline_data`
2. `step_local_2` runs and can see the data from `step_local`
3. Final result contains data from both steps

In [ ]:
print_header("Test 2: Mixed Steps with Data Flow")

main_pid = os.getpid()

print(f"\n  Main process PID: {main_pid}")
print(f"  Pipeline: {PIPELINE_FILES['mixed'].name}")

try:
    print("\n  Running pipeline...")
    result = run_pipeline(
        yaml_path=str(PIPELINE_FILES['mixed']),
        label="test_mixed",
        input_data={"test_input": "from_notebook"}
    )
    
    # Check results
    step1 = result.get("step_local", {})
    step2 = result.get("step_local_2", {})
    
    print(f"\n  Step Execution:")
    print(f"    step_local   - PID: {step1.get('process_id')}, Env: {step1.get('environment_name')}")
    print(f"    step_local_2 - PID: {step2.get('process_id')}, Env: {step2.get('environment_name')}")
    
    previous_steps = step2.get("previous_steps_found", [])
    print(f"\n  Data flow verification:")
    print(f"    step_local_2 found previous steps: {previous_steps}")
    print(f"    Input preserved: {result.get('input', {}).get('test_input')}")
    
    # Validate
    same_process = (step1.get('process_id') == main_pid and step2.get('process_id') == main_pid)
    data_flows = "step_local" in previous_steps
    input_ok = result.get('input', {}).get('test_input') == "from_notebook"
    
    success = same_process and data_flows and input_ok
    test_results['mixed'] = success
    
    details = []
    if same_process: details.append("same process ✓")
    if data_flows: details.append("data flows ✓")
    if input_ok: details.append("input preserved ✓")
    
    print_result("Mixed Steps", success, ", ".join(details))
    
except Exception as e:
    test_results['mixed'] = False
    print_result("Mixed Steps", False, str(e))

### 3.3 Test 3: Step-Level Environment Switching

This test verifies that individual steps can run in different Conda environments.

**What happens:**
1. `step_local` runs in main process
2. `step_env` has `environment: "env_a"` → runs in subprocess
3. `step_local_2` runs back in main process
4. Data flows through all steps despite environment switching

⚠️ **Requires:** Conda environments to be set up

In [ ]:
print_header("Test 3: Step-Level Environment Switching")

main_pid = os.getpid()
main_env = os.path.basename(sys.prefix)

print(f"\n  Main process PID: {main_pid}")
print(f"  Main environment: {main_env}")
print(f"  Pipeline: {PIPELINE_FILES['step_env'].name}")

try:
    print("\n  Running pipeline...")
    result = run_pipeline(
        yaml_path=str(PIPELINE_FILES['step_env']),
        label="test_step_env",
        input_data={}
    )
    
    # Check results
    step1 = result.get("step_local", {})
    step_env = result.get("step_env", {})
    step2 = result.get("step_local_2", {})
    
    print(f"\n  Step Execution:")
    print(f"    {'Step':<15} {'PID':<10} {'Environment':<15}")
    print(f"    {'-'*40}")
    print(f"    {'step_local':<15} {step1.get('process_id', 'N/A'):<10} {step1.get('environment_name', 'N/A'):<15}")
    print(f"    {'step_env':<15} {step_env.get('process_id', 'N/A'):<10} {step_env.get('actual_environment', 'N/A'):<15}")
    print(f"    {'step_local_2':<15} {step2.get('process_id', 'N/A'):<10} {step2.get('environment_name', 'N/A'):<15}")
    
    # Validate
    env_switched = step_env.get('process_id') != main_pid
    env_correct = step_env.get('environment_match', False)
    data_flows = len(step2.get('previous_steps_found', [])) >= 2
    
    print(f"\n  Validation:")
    print(f"    Environment switched (different PID): {'✓' if env_switched else '✗'}")
    print(f"    Correct environment: {'✓' if env_correct else '?'}")
    print(f"    Data flows through: {'✓' if data_flows else '✗'}")
    
    success = env_switched and data_flows
    test_results['step_env'] = success
    print_result("Step Environment Switching", success)
    
except Exception as e:
    test_results['step_env'] = False
    error_msg = str(e)
    if "conda" in error_msg.lower() or "environment" in error_msg.lower():
        print_result("Step Environment Switching", False, "Conda environment not available")
        print("\n  💡 Tip: Run the 'Create Test Environments' section first")
    else:
        print_result("Step Environment Switching", False, error_msg)

### 3.4 Test 4: Pipeline-Level Environment Switching

This test verifies that an entire pipeline can run in a specified Conda environment.

**What happens:**
1. YAML has `environment: "env_a"` in metadata
2. Engine detects this and spawns the entire pipeline in `env_a`
3. All steps run in the same subprocess environment

⚠️ **Requires:** Conda environments to be set up

In [ ]:
print_header("Test 4: Pipeline-Level Environment Switching")

main_pid = os.getpid()
main_env = os.path.basename(sys.prefix)

print(f"\n  Main process PID: {main_pid}")
print(f"  Main environment: {main_env}")
print(f"  Pipeline: {PIPELINE_FILES['pipeline_env'].name}")
print(f"  Expected: All steps run in 'env_a'")

try:
    print("\n  Running pipeline...")
    result = run_pipeline(
        yaml_path=str(PIPELINE_FILES['pipeline_env']),
        label="test_pipeline_env",
        input_data={}
    )
    
    # Check results
    step1 = result.get("step_local", {})
    step2 = result.get("step_local_2", {})
    
    print(f"\n  Step Execution:")
    print(f"    step_local   - Env: {step1.get('environment_name')}")
    print(f"    step_local_2 - Env: {step2.get('environment_name')}")
    
    # Validate
    same_env = step1.get('environment_name') == step2.get('environment_name')
    different_from_main = step1.get('environment_name') != main_env
    
    print(f"\n  Validation:")
    print(f"    Both steps in same environment: {'✓' if same_env else '✗'}")
    print(f"    Different from main: {'✓' if different_from_main else '(same as main)'}")
    
    success = same_env
    test_results['pipeline_env'] = success
    
    if success:
        if different_from_main:
            print_result("Pipeline Environment Switching", True, 
                        f"Pipeline ran in '{step1.get('environment_name')}'")
        else:
            print_result("Pipeline Environment Switching", True, 
                        "Already in target environment")
    else:
        print_result("Pipeline Environment Switching", False, "Steps in different environments")
    
except Exception as e:
    test_results['pipeline_env'] = False
    error_msg = str(e)
    if "conda" in error_msg.lower() or "environment" in error_msg.lower():
        print_result("Pipeline Environment Switching", False, "Conda environment not available")
        print("\n  💡 Tip: Run the 'Create Test Environments' section first")
    else:
        print_result("Pipeline Environment Switching", False, error_msg)

### 3.5 Test 5: Combined Environment Switching

This is the most complex test - nested environment switching.

**What happens:**
1. Pipeline has `environment: "env_a"` → spawns subprocess in env_a
2. `step_local` runs in env_a
3. `step_env_b` has `environment: "env_b"` → spawns nested subprocess
4. `step_local_2` runs back in env_a

```
Main Process (your kernel)
    └── Subprocess (env_a) - pipeline
            ├── step_local (env_a)
            ├── Subprocess (env_b)
            │       └── step_env_b
            └── step_local_2 (env_a)
```

⚠️ **Requires:** Both `env_a` and `env_b` to be set up

In [ ]:
print_header("Test 5: Combined Environment Switching")

main_pid = os.getpid()
main_env = os.path.basename(sys.prefix)

print(f"\n  Main process PID: {main_pid}")
print(f"  Main environment: {main_env}")
print(f"  Pipeline: {PIPELINE_FILES['combined'].name}")
print(f"\n  Expected flow:")
print(f"    main → env_a (pipeline) → env_b (step) → env_a")

try:
    print("\n  Running pipeline...")
    result = run_pipeline(
        yaml_path=str(PIPELINE_FILES['combined']),
        label="test_combined",
        input_data={}
    )
    
    # Check results
    step1 = result.get("step_local", {})
    step_b = result.get("step_env_b", {})
    step2 = result.get("step_local_2", {})
    
    print(f"\n  Execution Flow:")
    print(f"    {'Step':<15} {'PID':<10} {'Environment':<15}")
    print(f"    {'-'*45}")
    print(f"    {'(main)':<15} {main_pid:<10} {main_env:<15}")
    print(f"    {'step_local':<15} {step1.get('process_id', 'N/A'):<10} {step1.get('environment_name', 'N/A'):<15}")
    print(f"    {'step_env_b':<15} {step_b.get('process_id', 'N/A'):<10} {step_b.get('actual_environment', 'N/A'):<15}")
    print(f"    {'step_local_2':<15} {step2.get('process_id', 'N/A'):<10} {step2.get('environment_name', 'N/A'):<15}")
    
    # Validate
    env_a = step1.get('environment_name')
    env_b = step_b.get('actual_environment')
    
    pipeline_consistent = (step1.get('environment_name') == step2.get('environment_name'))
    step_b_isolated = (step_b.get('process_id') != step1.get('process_id'))
    envs_different = (env_a and env_b and env_a.lower() != env_b.lower())
    data_flows = len(step2.get('previous_steps_found', [])) >= 2
    
    print(f"\n  Validation:")
    print(f"    Pipeline steps consistent (A=A): {'✓' if pipeline_consistent else '✗'}")
    print(f"    step_env_b isolated: {'✓' if step_b_isolated else '✗'}")
    print(f"    Environments different (A≠B): {'✓' if envs_different else '?'}")
    print(f"    Data flows through all: {'✓' if data_flows else '✗'}")
    
    success = pipeline_consistent and step_b_isolated and data_flows
    test_results['combined'] = success
    
    if success:
        if envs_different:
            print_result("Combined Environment Switching", True,
                        f"Pipeline in '{env_a}', step switched to '{env_b}'")
        else:
            print_result("Combined Environment Switching", True,
                        "Isolation works (environments may be same)")
    else:
        print_result("Combined Environment Switching", False)
    
except Exception as e:
    test_results['combined'] = False
    error_msg = str(e)
    if "conda" in error_msg.lower() or "environment" in error_msg.lower():
        print_result("Combined Environment Switching", False, "Conda environment not available")
        print("\n  💡 Tip: Run the 'Create Test Environments' section first")
    else:
        print_result("Combined Environment Switching", False, error_msg)

### 3.6 Run All Tests

Run this cell to execute all tests in sequence and see a summary.

In [ ]:
# Summary of test results
print_header("Test Summary")

print("\n  Results:")
print("  " + "-" * 40)

for test_name, passed in test_results.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"    {status}: {test_name}")

print("  " + "-" * 40)

passed_count = sum(1 for v in test_results.values() if v)
total_count = len(test_results)

print(f"\n  Total: {passed_count}/{total_count} tests passed")

if passed_count == total_count:
    print("\n  🎉 ALL TESTS PASSED!")
elif passed_count >= 2:
    print("\n  ⚠ Some tests failed (environment tests may need Conda setup)")
else:
    print("\n  ❌ Multiple tests failed - check configuration")

---

## 4. Cleanup

When you're done testing, you can remove the test environments to free up disk space.

### 4.1 Check What Will Be Removed

First, let's see which environments exist and how much space they use.

In [ ]:
def get_env_size(env_name: str) -> str:
    """Estimate the size of a Conda environment."""
    conda_info = get_conda_info()
    if not conda_info:
        return "unknown"
    
    # Find environment path
    env_path = None
    for path in conda_info.get('envs', []):
        if Path(path).name == env_name:
            env_path = path
            break
    
    if not env_path or not os.path.isdir(env_path):
        return "not found"
    
    # Calculate size
    try:
        total = 0
        for dirpath, dirnames, filenames in os.walk(env_path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.exists(fp):
                    total += os.path.getsize(fp)
        
        if total > 1024 * 1024 * 1024:
            return f"{total / (1024**3):.1f} GB"
        elif total > 1024 * 1024:
            return f"{total / (1024**2):.0f} MB"
        else:
            return f"{total / 1024:.0f} KB"
    except:
        return "unknown"

def remove_environment(env_name: str) -> bool:
    """Remove a Conda environment."""
    print(f"  Removing {env_name}...", end=" ", flush=True)
    
    if not env_exists(env_name):
        print("(not found, skipping)")
        return True
    
    result = subprocess.run(
        ["conda", "env", "remove", "-n", env_name, "-y"],
        capture_output=True, text=True
    )
    
    if result.returncode == 0:
        print("✓ removed")
        return True
    else:
        print("✗ failed")
        if result.stderr:
            print(f"    Error: {result.stderr[:100]}")
        return False

# Show current status
print_header("Environments to Remove")
print("\n  Current status:")
print("  " + "-" * 40)

total_size = 0
for env_name in ENVIRONMENTS:
    exists = env_exists(env_name)
    if exists:
        size = get_env_size(env_name)
        print(f"  ✓ {env_name}: {size}")
    else:
        print(f"  ✗ {env_name}: not found")

print("  " + "-" * 40)
print("\n  Run the next cell to remove these environments.")

### 4.2 Remove Environments

**⚠️ Warning:** This will permanently delete the test environments!

To proceed:
1. Change `CONFIRM_CLEANUP = False` to `CONFIRM_CLEANUP = True`
2. Run the cell

In [ ]:
# ============================================================
# SAFETY FLAG - Change to True to enable cleanup
# ============================================================
CONFIRM_CLEANUP = False  # <-- Change to True to remove environments
# ============================================================

if CONFIRM_CLEANUP:
    print_header("Removing Test Environments")
    print()
    
    success_count = 0
    for env_name in ENVIRONMENTS:
        if remove_environment(env_name):
            success_count += 1
    
    print()
    print("  " + "=" * 40)
    print(f"  Removed: {success_count}/{len(ENVIRONMENTS)} environments")
    print("  " + "=" * 40)
    
    if success_count == len(ENVIRONMENTS):
        print("\n  ✓ Cleanup complete! Disk space freed.")
    else:
        print("\n  ⚠ Some environments could not be removed.")
        print("    Try closing any terminals using these environments.")
else:
    print("🔒 Cleanup is DISABLED (safety lock)")
    print()
    print("   To remove the test environments:")
    print("   1. Change CONFIRM_CLEANUP = False to CONFIRM_CLEANUP = True")
    print("   2. Run this cell again")
    print()
    print("   This will remove: env_a, env_b, env_c")

### 4.3 Verify Cleanup

Run this cell to confirm the environments have been removed.

In [ ]:
# Verify environments are removed
print_header("Post-Cleanup Status")
print()

remaining = []
for env_name in ENVIRONMENTS:
    if env_exists(env_name):
        print(f"  ⚠ {env_name}: still exists")
        remaining.append(env_name)
    else:
        print(f"  ✓ {env_name}: removed")

print()
if remaining:
    print(f"  {len(remaining)} environment(s) still exist.")
else:
    print("  ✓ All test environments have been removed.")

---

## 5. Troubleshooting

### Common Issues

#### "Conda not found"
- Make sure Conda is installed
- Run this notebook from a Conda-enabled terminal/environment
- On Windows, use Anaconda Prompt

#### "Environment not found" during tests
- Run Section 2.2 to create test environments
- Check that environments exist with `conda env list`

#### Tests 1 & 2 pass but 3-5 fail
- This is expected if Conda environments aren't set up
- Local execution works without Conda
- Environment switching requires the test environments

#### Import errors
- Make sure you're running this notebook from the `scripts/` directory
- Check that `engine.py` exists in the `engine/` folder

### Debug Information

In [ ]:
# Print debug information
print_header("Debug Information")

print(f"\n  Python:")
print(f"    Version: {sys.version}")
print(f"    Executable: {sys.executable}")
print(f"    Prefix: {sys.prefix}")

print(f"\n  Paths:")
print(f"    Working dir: {Path.cwd()}")
print(f"    Engine dir: {ENGINE_DIR}")
print(f"    Engine exists: {(ENGINE_DIR / 'engine.py').exists()}")

print(f"\n  Conda:")
conda_info = get_conda_info()
if conda_info:
    print(f"    Available: Yes")
    print(f"    Version: {conda_info.get('conda_version')}")
    print(f"    Environments: {len(conda_info.get('envs', []))}")
else:
    print(f"    Available: No")

print(f"\n  Test Results:")
for name, result in test_results.items():
    print(f"    {name}: {'Pass' if result else 'Fail'}")

---

## Summary

This notebook provides interactive testing for the Pipeline Engine:

| Section | Purpose |
|---------|--------|
| **Setup** | Import engine, define helpers |
| **Environment Management** | Create/check/remove Conda environments |
| **Tests 1-2** | Local execution (no Conda required) |
| **Tests 3-5** | Environment switching (requires Conda) |
| **Cleanup** | Remove test environments |

For more information, see the [Pipeline Engine Documentation](../../../docs/Pipeline_Engine_Documentation.md).